In [23]:
from google.cloud import storage
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import load_model
import cv2
import os
import shutil
import numpy as np
from tqdm.notebook import tqdm
from firebase_admin import credentials, storage

## Download video uploaded on firebase storage

In [24]:
import os
from google.cloud import storage

# Name of the bucket on firebase
bucket_name = "remembron.appspot.com"

# Path to local file where the videos are to be stored
local_directory = r"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\DataSet5_Trial"

# Firebase service account key file
credentials_file = r"C:\Users\SHARVIN JOSHI\Downloads\remembron-firebase-adminsdk-pd0su-80c4172881.json"

# Creating a storage client object
storage_client = storage.Client.from_service_account_json(credentials_file)

# Reference to bucket
bucket = storage_client.get_bucket(bucket_name)

# Listing all the blobs in the bucket
blobs = bucket.list_blobs()

# Dictionary to store the information in {name : information (age , relation with user etc)} format
dict = {}

for blob in blobs:        
    try:
        local_file_path = os.path.join(local_directory, blob.name)

        # Downloading to local file
        blob.download_to_filename(local_file_path)

        # Print the name of the downloaded file
        print(f"File downloaded to: {local_file_path}")
        
        s1 = (local_file_path.split("\\"))[-1].split(".")
        s2 = s1[0].split("_")
        
        dict.update({s2[0]:[s2[1],s2[2]]})
    except:
        continue
print(dict)

File downloaded to: C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\DataSet5_Trial\Arnav_20_Uncle.mp4
File downloaded to: C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\DataSet5_Trial\Nikhil_33_Father.mp4
File downloaded to: C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\DataSet5_Trial\Ninad_10_Son.mp4
File downloaded to: C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\DataSet5_Trial\Sharvin_21_Brother.mp4
{'Arnav': ['20', 'Uncle'], 'Nikhil': ['33', 'Father'], 'Ninad': ['10', 'Son'], 'Sharvin': ['21', 'Brother']}


## Making classes and extracting training images from the videos

In [25]:
# Using haarcascading method to identify faces in images

# Importing face cascade
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

# Setting the path where folders should be stored
dataset_path = r"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\DataSet5_Trial"

os.chdir(dataset_path)
print(os.listdir(dataset_path))

# Creating classes for training dataset and storing the face images in respective classes
# Name of the class is the name of the person whose image is stored inside it
for x in os.listdir(dataset_path):
    os.mkdir(dataset_path +"\\" + (x.split("_"))[0])
    destination_directory = dataset_path +"\\" + (x.split("_"))[0]
    source = rf"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\DataSet5_Trial\{x}"
    shutil.copy2(source, destination_directory)
    os.remove(source)

parent_path = r"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\DataSet5_Trial"

os.chdir(parent_path)
print(os.listdir())

# Going frame by frame in the video and identifying the face in each frame and storing it to the respective class
for i in os.listdir():
    path = parent_path + "\\" + i
    os.chdir(path)
    print(os.listdir())

    video_path = path + "\\" + (os.listdir())[0]
    print(video_path)

    cap = cv2.VideoCapture(video_path)     # Capturing the video taken from Firebase
    a = 1
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        faces = face_cascade.detectMultiScale(gray, 1.3, 5)

        for (x, y, w, h) in faces:
            cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)   # Making a rectangle around the face,(x,y) being the top left coordinate 

        for (x, y, w, h) in faces:
            roi_color = frame[y:y+h, x:x+w]
            file_name = f"v{a}.jpg"             # Saving each image with a unique name
            b = parent_path + "\\" + i
            full_file_path = os.path.join(b, file_name)
            roi_color_resized = cv2.resize(roi_color,(128,128))
            cv2.imwrite(full_file_path, roi_color_resized)
            a += 1

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

['Arnav_20_Uncle.mp4', 'Nikhil_33_Father.mp4', 'Ninad_10_Son.mp4', 'Sharvin_21_Brother.mp4']
['Arnav', 'Nikhil', 'Ninad', 'Sharvin']
['Arnav_20_Uncle.mp4']
C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\DataSet5_Trial\Arnav\Arnav_20_Uncle.mp4


error: OpenCV(4.9.0) D:\a\opencv-python\opencv-python\opencv\modules\highgui\src\window.cpp:1338: error: (-2:Unspecified error) The function is not implemented. Rebuild the library with Windows, GTK+ 2.x or Cocoa support. If you are on Ubuntu or Debian, install libgtk2.0-dev and pkg-config, then re-run cmake or configure script in function 'cvWaitKey'


## Training the model

In [9]:
# Loading the dataset 
dataset = tf.keras.utils.image_dataset_from_directory(r"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\DataSet5_Trial",image_size = (128,128),batch_size=32)

(train_images, train_labels) = next(iter(dataset))

Found 439 files belonging to 4 classes.


In [10]:
# Getting the class names 

class_names = dataset.class_names
class_names

['Arnav', 'Nikhil', 'Ninad', 'Sharvin']

In [21]:
# Architecture of the Convolutional Neural Netowrk (CNN)

# Initialising a Sequential model from Tensorflow Keras
model = tf.keras.models.Sequential()

# Adding the first filter layer having the activation function ReLU and a max pooling layer
model.add(tf.keras.layers.Conv2D(filters=32,kernel_size=(4, 4),strides=(1, 1),padding='valid',activation='relu',input_shape=(128,128,3)))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid'))

# Second layer of filter and max pool
model.add(tf.keras.layers.Conv2D(filters=16,kernel_size=(4, 4),strides=(1, 1),padding='valid',activation='relu'))
model.add(tf.keras.layers.MaxPooling2D(pool_size=(2, 2), strides=(2, 2), padding='valid'))

# Flattening (converting 3D numpy array to 1D array)
model.add(tf.keras.layers.Flatten())

# Adding the dense layers
model.add(tf.keras.layers.Dense(128, activation='relu'))
model.add(tf.keras.layers.Dense(64, activation='relu'))
model.add(tf.keras.layers.Dense(9, activation='softmax'))

In [22]:
# Compiling the model using adam optimizer and sparse cross entropy loss function
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

In [13]:
# Training the model
model.fit(train_images, train_labels, epochs=17, validation_split=0.2)

Epoch 1/17


1/1 [==============================] - 1s 1s/step - loss: 43.3079 - accuracy: 0.3200 - val_loss: 195.7690 - val_accuracy: 0.5714
Epoch 2/17
1/1 [==============================] - 0s 101ms/step - loss: 323.1032 - accuracy: 0.3200 - val_loss: 207.3475 - val_accuracy: 0.2857
Epoch 3/17
1/1 [==============================] - 0s 96ms/step - loss: 229.3368 - accuracy: 0.2000 - val_loss: 111.1950 - val_accuracy: 0.0000e+00
Epoch 4/17
1/1 [==============================] - 0s 93ms/step - loss: 113.4063 - accuracy: 0.1600 - val_loss: 26.2348 - val_accuracy: 0.2857
Epoch 5/17
1/1 [==============================] - 0s 107ms/step - loss: 30.4189 - accuracy: 0.2800 - val_loss: 1.4369 - val_accuracy: 0.5714
Epoch 6/17
1/1 [==============================] - 0s 100ms/step - loss: 2.4222 - accuracy: 0.6000 - val_loss: 1.0096 - val_accuracy: 0.5714
Epoch 7/17
1/1 [==============================] - 0s 102ms/step - loss: 1.2026 - accuracy: 0.5600 - val_loss: 0.9216 - val_accuracy: 0.7143
Epoc

In [14]:
# Saving the trained model
from tensorflow.keras.models import save_model
save_model(model, r"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\model2.h5")

C:\Users\SHARVIN JOSHI\AppData\Local\Temp\ipykernel_19504\344929621.py:2: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  save_model(model, r"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\model2.h5")


In [17]:
import firebase_admin
import google.cloud

## Getting the input image which we need to predict

In [18]:
# Loading the bucket name, local directory to store the image and the credential file
bucket_name = "remembron.appspot.com"
local_directory = r"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\Prediction"
credentials_file = r"C:\Users\SHARVIN JOSHI\Downloads\remembron-firebase-adminsdk-pd0su-80c4172881.json"

storage_client = storage.Client.from_service_account_json(credentials_file)

bucket = storage_client.get_bucket(bucket_name)

blobs = bucket.list_blobs()

for blob in blobs:
    try:
        local_file_path = os.path.join(local_directory, os.path.basename(blob.name))
        blob.download_to_filename(local_file_path)
        print(f"File downloaded to: {local_file_path}")
    except Exception as e:
        print(f"Error downloading file: {e}")


File downloaded to: C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\Prediction\Arnav_20_Uncle.mp4
File downloaded to: C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\Prediction\Nikhil_33_Father.mp4
File downloaded to: C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\Prediction\Ninad_10_Son.mp4
File downloaded to: C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\Prediction\Sharvin_21_Brother.mp4
Error downloading file: [Errno 2] No such file or directory: 'C:\\Users\\SHARVIN JOSHI\\Desktop\\P\\FACE RECOGNITION\\Prediction\\'
File downloaded to: C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\Prediction\my_uploaded_image.jpg


## Giving the model the image for prediction

In [19]:
model = load_model(r"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\model2.h5")

def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(128, 128))  # adjust target_size according to your model
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    return x
class_labels = class_names

os.chdir(r"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\Prediction")
img_path = rf"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\Prediction\{(os.listdir())[0]}"
img = preprocess_image(img_path)

predictions = model.predict(img)
print("Predictions:")
scores = []
for i, score in enumerate(predictions[0]):
    scores.append(score)
print(class_labels[scores.index(max(scores))])

1/1 [==============================] - 0s 66ms/step
Predictions:
Nikhil


In [16]:
model = load_model(r"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\model2.h5")

def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(128, 128))  # adjust target_size according to your model
    x = image.img_to_array(img)
    x = np.expand_dims(x, axis=0)
    return x
class_labels = class_names
positive=0
os.chdir(r"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\DataSet5_Trial\Arnav")

for img_path in tqdm(os.listdir(r"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\DataSet5_Trial\Arnav")):

    img = preprocess_image(img_path)

    predictions = model.predict(img)

    scores = []
    for i, score in enumerate(predictions[0]):
        scores.append(score)
    label = class_labels[scores.index(max(scores))]
    if label=='Arnav':
        positive +=1
print(100*(positive/len(os.listdir(r"C:\Users\SHARVIN JOSHI\Desktop\P\FACE RECOGNITION\DataSet5_Trial\Arnav"))))

  0%|          | 0/141 [00:00<?, ?it/s]

1/1 [==============================] - 0s 18ms/step
97.87234042553192
